In [3]:
import pandas as pd
import numpy as np
import os

In [5]:
# Step 1: Create dataframe df_target from 'raw' data
data_raw = {
    'cust_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'rule_10': [1, 1, 1, 0, 0, 0, 1, 1, 0, 0],
    'rule_15': [0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
    'rule_18': [0, 1, 0, 1, 0, 1, 0, 1, 0, 1],
    'rule_20': [1, 1, 1, 0, 0, 0, 0, 1, 0, 1]
}
df_target = pd.DataFrame(data_raw)
print(df_target.head(100))

   cust_id  rule_10  rule_15  rule_18  rule_20
0        1        1        0        0        1
1        2        1        0        1        1
2        3        1        0        0        1
3        4        0        0        1        0
4        5        0        0        0        0
5        6        0        1        1        0
6        7        1        1        0        0
7        8        1        0        1        1
8        9        0        0        0        0
9       10        0        0        1        1


In [15]:
# # Step 2: Create dataframe df_key from 'key' data
# data_key = {
#     'rule': ['rule_10', 'rule_15', 'rule_18', 'rule_20'],
#     'exclusion_reason': ['invalid ID', 'Foreign', 'No credit card', 'Wrong promo code']
# }
# df_key = pd.DataFrame(data_key)
# print(df_key.head(100))

In [17]:
# --- 2. Create df_key from a dictionary ---
# As requested, df_key is now created from a Python dictionary.
key_data_dict = {
    'rule': ['rule_10', 'rule_15', 'rule_18', 'rule_20'],
    'exclusion_reason': ['invalid ID', 'Foreign', 'No credit card', 'Wrong promo code']
}

df_key = pd.DataFrame(key_data_dict)
print(df_key.head(100))

      rule  exclusion_reason
0  rule_10        invalid ID
1  rule_15           Foreign
2  rule_18    No credit card
3  rule_20  Wrong promo code


In [16]:
# --- 2. Create df_key from the 'key' data ---
import io

key_data = """rule,exclusion_reason
rule_10,invalid ID
rule_15,Foreign
rule_18,No credit card
rule_20,Wrong promo code
"""

df_key = pd.read_csv(io.StringIO(key_data))
print(df_key.head(100))

      rule  exclusion_reason
0  rule_10        invalid ID
1  rule_15           Foreign
2  rule_18    No credit card
3  rule_20  Wrong promo code


In [10]:
# --- 3. Generate the summary table ---

# Get the list of rule columns in the specified order from df_key
rule_columns = df_key['rule'].tolist()

In [11]:
print(rule_columns)

['rule_10', 'rule_15', 'rule_18', 'rule_20']


In [12]:
summary_data = []

# This list will keep track of the rules to check for the waterfall calculation
waterfall_check_rules = []

for rule in rule_columns:
    # Logic for n_cust: Count of customers where the rule is 1.
    n_cust = df_target[rule].sum()

    # Add the current rule to the list for the waterfall check
    waterfall_check_rules.append(rule)
    
    # Logic for waterfall: Count of customers where all rules up to this point are 0.
    # We create a boolean mask. For each customer (row), it checks if all values
    # in the 'waterfall_check_rules' columns are equal to 0.
    mask = (df_target[waterfall_check_rules] == 0).all(axis=1)
    
    # The waterfall count is the sum of rows where the mask is True.
    waterfall = mask.sum()

    summary_data.append({
        'rule': rule,
        'n_cust': n_cust,
        'waterfall': waterfall
    })

In [13]:
# Create a DataFrame from the calculated summary data
df_summary = pd.DataFrame(summary_data)

# Merge with the key dataframe to add the 'exclusion_reason' column
df_result = pd.merge(df_key, df_summary, on='rule')

# Reorder columns to match the expected output format from the 'result' worksheet [[doc_1]]
df_result = df_result[['rule', 'exclusion_reason', 'n_cust', 'waterfall']]


In [14]:
# --- Display the final result ---
print("--- df_target ---")
print(df_target)
print("\n--- df_key ---")
print(df_key)
print("\n--- Final Result ---")
print(df_result)

--- df_target ---
   cust_id  rule_10  rule_15  rule_18  rule_20
0        1        1        0        0        1
1        2        1        0        1        1
2        3        1        0        0        1
3        4        0        0        1        0
4        5        0        0        0        0
5        6        0        1        1        0
6        7        1        1        0        0
7        8        1        0        1        1
8        9        0        0        0        0
9       10        0        0        1        1

--- df_key ---
      rule  exclusion_reason
0  rule_10        invalid ID
1  rule_15           Foreign
2  rule_18    No credit card
3  rule_20  Wrong promo code

--- Final Result ---
      rule  exclusion_reason  n_cust  waterfall
0  rule_10        invalid ID       5          5
1  rule_15           Foreign       2          4
2  rule_18    No credit card       5          2
3  rule_20  Wrong promo code       5          2


In [19]:
# --- Display the final sentence as requested ---

# Get the last value from the 'waterfall' column
eligible_customers = df_result['waterfall'].iloc[-1]

# Print the final sentence using an f-string with comma formatting
# The ':, ' specifier tells Python to use a comma as a thousands separator.
print(f"No. of eligible customers = {eligible_customers:,}")

No. of eligible customers = 2
